In [3]:
from google.colab import files
files.upload()  # Upload kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{\r\n  "username": "chaitanyadeore",\r\n  "key": "KGAT_907808ff718796b9e57c26d023bc468d"\r\n}'}

In [4]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 600)

In [5]:
!pip install kaggle

!kaggle datasets download -d kienngx/nemotron-30b-competition-trainingdata-cot-labels
!unzip nemotron-30b-competition-trainingdata-cot-labels.zip

Dataset URL: https://www.kaggle.com/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels
License(s): apache-2.0
100% 3.72M/3.72M [00:00<00:00, 137MB/s]

Archive:  nemotron-30b-competition-trainingdata-cot-labels.zip
  inflating: final_Nemotron_training_data.csv  


In [12]:
import pandas as pd

df = pd.read_csv("final_Nemotron_training_data.csv")

print(df.head())
print(df.columns)

         id                                             prompt  \
0  00066667  In Alice's Wonderland, a secret bit manipulati...   
1  000b53cf  In Alice's Wonderland, a secret bit manipulati...   
2  00189f6a  In Alice's Wonderland, secret encryption rules...   
3  001b24c4  In Alice's Wonderland, numbers are secretly co...   
4  001c63cb  In Alice's Wonderland, secret encryption rules...   

                  answer                                      generated_cot  \
0               10010111  The transformation seems complex and non-linea...   
1               01000011  The task is to deduce a bit manipulation rule ...   
2      cat imagines book  The task is to decrypt the text "trb wzrswvog ...   
3                XXXVIII  The task is to convert 38 to Roman numerals ba...   
4  wizard creates secret  The task is to decrypt "hncreo learpaq qaleap"...   

                                            label  
0         bitwise and binary transformation tasks  
1         bitwise and bi

In [15]:
print(df.columns)

Index(['id', 'prompt', 'answer', 'generated_cot', 'label'], dtype='object')


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = df.fillna("")

df["text"] = df["prompt"] + " " + df["generated_cot"] + " " + df["answer"]

le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=3000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [18]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=200)
lr.fit(X_train_vec, y_train)

print("Logistic Regression Accuracy:", lr.score(X_test_vec, y_test))

Logistic Regression Accuracy: 0.9278947368421052


In [19]:
from sklearn.svm import SVC

svm = SVC()
svm.fit(X_train_vec, y_train)

print("SVM Accuracy:", svm.score(X_test_vec, y_test))

SVM Accuracy: 0.9152631578947369


In [20]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train_vec, y_train)

print("Random Forest Accuracy:", rf.score(X_test_vec, y_test))

Random Forest Accuracy: 0.9284210526315789


In [21]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_vec, y_train)

print("Naive Bayes Accuracy:", nb.score(X_test_vec, y_test))

Naive Bayes Accuracy: 0.9010526315789473
